# Serialise Detections to a JSON File

---

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Child-pi/supervision/blob/develop/docs/notebooks/serialise-detections-to-json.ipynb)

This cookbook introduces the [sv.JSONSink](https://supervision.roboflow.com/develop/detection/tools/save_detections/#supervision.detection.tools.json_sink.JSONSink) tool designed to write captured object detection data to file from video streams/file.

Click the `Open in Colab` button to run the cookbook on Google Colab.

In [ ]:
!pip install -q rfdetr supervision trackers pandas tqdm


In [ ]:
import json
from collections import defaultdict
from typing import List

import numpy as np
import pandas as pd
from rfdetr import RFDETRSmall
import supervision as sv
from supervision.assets import VideoAssets, download_assets
from tqdm import tqdm
from trackers import ByteTrackTracker


The parameters defined below are:
* `SOURCE_VIDEO_PATH` - the path to the input video
* `CONFIDENCE_THRESHOLD` - do not include detections below this confidence level
* `FILE_NAME` - write the JSON output to this file


In [ ]:
SOURCE_VIDEO_PATH = download_assets(VideoAssets.PEOPLE_WALKING)
CONFIDENCE_THRESHOLD = 0.3
FILE_NAME = "detections.json"


As a result of executing the above `download_assets(VideoAssets.PEOPLE_WALKING)` , you will download a video file and save it at the `SOURCE_VIDEO_PATH`. Keep in mind that the video preview below works only in the web version of the cookbooks and not in Google Colab.

<video controls width="1280">
  <source src="https://media.roboflow.com/supervision/video-examples/people-walking.mp4" type="video/mp4">
</video>


## Read single frame from video

The [`get_video_frames_generator`](https://supervision.roboflow.com/develop/utils/video/#supervision.utils.video.get_video_frames_generator) enables us to easily iterate over video frames. Let's create a video generator for our sample input file and display its first frame on the screen.


In [ ]:
generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(generator)

sv.plot_image(frame, (12, 12))


We can also use [`VideoInfo.from_video_path`](https://supervision.roboflow.com/develop/utils/video/#supervision.utils.video.VideoInfo) to learn basic information about our video, such as duration, resolution, or FPS.

In [ ]:
sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)


## Initialize ByteTrackTracker

[ByteTrackTracker](https://trackers.roboflow.com/latest/) is a multi-object tracking algorithm from the external `trackers` package. It tracks and links detected objects across multiple frames, providing consistent IDs for each object. Initialize the `ByteTrackTracker` object.

In [ ]:
byte_track = ByteTrackTracker(
    minimum_consecutive_frames=3, track_activation_threshold=CONFIDENCE_THRESHOLD
)
byte_track.reset()


## Process video and save detections to JSON file

To save detections to a JSON file, we use [`sv.JSONSink`](https://supervision.roboflow.com/latest/how_to/save_detections/#save-detections-as-json). We iterate over the video frames using [`sv.get_video_frames_generator`](https://supervision.roboflow.com/develop/utils/video/#supervision.utils.video.get_video_frames_generator), run object detection using `RFDETRSmall`, maintain persistent object IDs using `ByteTrackTracker`, and append detections along with custom metadata like `frame_number` to `json_sink`.

In [ ]:
model = RFDETRSmall()
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)

with sv.JSONSink(FILE_NAME) as json_sink:
    for frame_number, frame in enumerate(tqdm(generator, total=video_info.total_frames)):
        # Model prediction: convert BGR frame to RGB
        detections = model.predict(frame[:, :, ::-1].copy(), threshold=CONFIDENCE_THRESHOLD)

        # Filter for person detections
        detections = detections[detections.data["class_name"] == "person"]

        # Update tracking IDs
        detections = byte_track.update(detections)
        detections = detections[detections.tracker_id != -1]

        # Write detections to JSON sink
        json_sink.append(detections, custom_data={"frame_number": frame_number})


## Visualize results of detections JSON data with Pandas

Let's inspect our resulting data using Pandas.


In [ ]:
df = pd.read_json(FILE_NAME)
df


## Convert JSON data to sv.Detections

We can parse the serialised JSON rows back into `sv.Detections` objects grouped by frame.

In [ ]:
def json_to_detections(json_file: str) -> List[sv.Detections]:
    rows_by_frame_number = defaultdict(list)
    with open(json_file, "r") as f:
        data = json.load(f)
    for row in data:
        frame_number = int(row["frame_number"])
        rows_by_frame_number[frame_number].append(row)

    detections_list = []
    for frame_number, rows in rows_by_frame_number.items():
        xyxy = []
        class_id = []
        confidence = []
        tracker_id = []
        custom_data = defaultdict(list)

        for row in rows:
            xyxy.append([float(row[key]) for key in ["x_min", "y_min", "x_max", "y_max"]])
            class_id.append(int(row["class_id"]) if row["class_id"] is not None and row["class_id"] != "" else 0)
            confidence.append(float(row["confidence"]) if row["confidence"] is not None and row["confidence"] != "" else 1.0)
            tracker_id.append(int(row["tracker_id"]) if row["tracker_id"] is not None and row["tracker_id"] != "" else -1)

            for custom_key in row.keys():
                if custom_key in ["x_min", "y_min", "x_max", "y_max", "class_id", "confidence", "tracker_id"]:
                    continue
                custom_data[custom_key].append(row[custom_key])

        detections_list.append(
            sv.Detections(
                xyxy=np.array(xyxy, dtype=np.float32),
                class_id=np.array(class_id, dtype=int),
                confidence=np.array(confidence, dtype=np.float32),
                tracker_id=np.array(tracker_id, dtype=int),
                data=dict(custom_data)
            )
        )
    
    return detections_list


In [ ]:
detections_list = json_to_detections(FILE_NAME)

print(f"Frames with detections: {len(detections_list)}")
print(detections_list[0])


### Annotate Frame

Visualize frame 100 alongside the detections loaded from the JSON file.

In [ ]:
FRAME_NUMBER = 100

detections = detections_list[FRAME_NUMBER]
frame_number = int(detections.data["frame_number"][0])

generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, start=frame_number)
frame = next(generator)


### Annotate Image with Detections

Finally, we annotate the frame with [`sv.BoxAnnotator`](https://supervision.roboflow.com/latest/detection/annotators/#supervision.annotators.core.BoxAnnotator) and [`sv.LabelAnnotator`](https://supervision.roboflow.com/latest/detection/annotators/#supervision.annotators.core.LabelAnnotator).

In [ ]:
bounding_box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

labels = [
    f"#{tracker_id} {class_name}"
    for tracker_id, class_name in zip(detections.tracker_id, detections.data["class_name"])
]

annotated_frame = frame.copy()
annotated_frame = bounding_box_annotator.annotate(scene=annotated_frame, detections=detections)
annotated_frame = label_annotator.annotate(scene=annotated_frame, detections=detections, labels=labels)
sv.plot_image(annotated_frame, (12, 12))


## References 📚

* Supervision: https://supervision.roboflow.com
* sv.Detections: https://supervision.roboflow.com/develop/detection/core/#detections
* Save Detections to JSON: https://supervision.roboflow.com/develop/how_to/save_detections/#save-detections-as-json
* Custom fields: https://supervision.roboflow.com/develop/how_to/save_detections/#custom-fields
* ByteTrack (external `trackers` package): https://trackers.roboflow.com/latest/
* RF-DETR: https://github.com/roboflow/rf-detr
